# Zero-Trust Privacy-Aware Agent Runtime: Multimodal Memory Gate
**Authors:** Yilin Pan & Tyler Hudgins | **Date:** April 2026

This notebook implements a strictly local, "zero-trust" privacy gate for a multimodal RAG agent. It intercepts text and images before they are embedded into the agent's memory (GCP Vector DB) to ensure sensitive personal information (SPI) is safely redacted.

**Key Architecture Features:**
1. **Zero Cloud APIs:** All inference, including text PII detection and image generation, runs locally.
2. **Ephemeral OCR:** OCR is used *only* to locate the bounding boxes of sensitive text in an image. The OCR strings are immediately deleted and never embedded into the vector database.
3. **Generative Inpainting:** Instead of applying heavy black boxes or blurs that destroy the semantic utility of an image, sensitive objects and text are seamlessly erased and replaced with a neutral background using Stable Diffusion.

In [ ]:
%pip install easyocr transformers diffusers torch torchvision matplotlib pillow accelerate

import os
import sys
import torch
import easyocr
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
from transformers import OwlViTProcessor, OwlViTForObjectDetection
from transformers import CLIPProcessor, CLIPModel, pipeline
from diffusers import AutoPipelineForInpainting

In [ ]:
# Ensure project root is on path regardless of how the notebook is launched
_nb_dir = os.path.abspath("")
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

from src.privacy.box_utils import (
    nms_boxes,
    expand_box,
    calculate_box_area,
    draw_boxes_on_image,
)

## Phase 0: Model Initialization

To maintain our zero-trust constraint, we load all necessary foundation models into local memory. 
* **Text PII:** `openai-privacy-filter` (Local open-weights model).
* **Text Localization:** `easyocr` (Used purely for spatial bounding boxes).
* **Object Detection:** `owlvit-base-patch32` (Zero-shot visual detector).
* **Generative Redaction:** `stable-diffusion-inpainting` (Loaded in float16 for efficiency).
* **Vectorization:** `clip-vit-base-patch32`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading local models onto {device}...")

# 1. Text PII Filter
pii_filter = pipeline(
    task="token-classification",
    model="openai/privacy-filter",
    aggregation_strategy="simple",
    device=0 if device == "cuda" else -1,
)

# 2. Ephemeral OCR Reader (GPU-accelerated when available)
reader = easyocr.Reader(['en'], gpu=(device == 'cuda'))

# 3. OwlViT zero-shot visual object detector
owl_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device)

# 4. Generative inpainting — fp16 variant only available for CUDA
_inpaint_kwargs = {"torch_dtype": torch.float16, "variant": "fp16"} if device == "cuda" \
                  else {"torch_dtype": torch.float32}
inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    **_inpaint_kwargs,
).to(device)

# 5. CLIP for final safe vectorization
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

print("All local models loaded successfully.")

## Phase 1 & 2: Sensing Gate and Local Privacy Detection

Two functions implement the detection layer:

**`redact_text(text, filter_pipe)`** — replaces PII spans in a string with `[ENTITY_TYPE]` tags.  
Defined first so `detect_privacy_risks_from_image` can call it for OCR results.

**`detect_privacy_risks_from_image(image, use_ocr=True)`** — returns deduplicated sensitive boxes:
1. **OwlViT object detection**: zero-shot queries for faces, passports, credit cards, etc.
2. **Ephemeral OCR** (when `use_ocr=True`): extracts text coordinates, classifies sensitivity,
   saves bounding boxes only — the OCR strings are deleted immediately and never embedded.
3. **Post-processing**: each box is padded by 8 px, then overlapping boxes are merged via
   IoU-based NMS (`src/privacy/box_utils.py`), reducing redundant inpainting regions.

**`detect_privacy_risks(user_text, image_path, use_ocr=True)`** — orchestrates both modalities
and returns a list of per-page result dicts covering all five input combinations:

| Input | OCR | Behaviour |
|---|---|---|
| text + image | ✓ | text PII + visual detection + OCR boxes |
| text + image | ✗ | text PII + visual detection only |
| text only | — | text PII, no image processing |
| image only | ✓ | visual detection + OCR boxes |
| image only | ✗ | visual detection only |

In [3]:
%pip install pdf2image
import os
from pdf2image import convert_from_path
from PIL import Image, ImageDraw

def load_images_from_path(file_path, dpi=200):
    """
    Load an image file or PDF file.

    Returns:
        images: list of PIL RGB images
    """

    if file_path is None or not str(file_path).strip():
        return []

    file_path = str(file_path)

    if not os.path.exists(file_path):
        print(f"[WARNING] File path does not exist: {file_path}")
        return []

    ext = os.path.splitext(file_path)[1].lower()

    # PDF input
    if ext == ".pdf":
        pages = convert_from_path(file_path, dpi=dpi)
        images = [page.convert("RGB") for page in pages]
        return images

    # Normal image input
    image = Image.open(file_path).convert("RGB")
    return [image]


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def redact_text(text: str, filter_pipe) -> tuple[str, bool]:
    """Replace PII spans with [ENTITY_TYPE] tags.

    Works backwards through spans so earlier character offsets stay valid
    after each substitution.
    """
    if text is None:
        return "", False
    text = str(text)
    if not text.strip():
        return text, False

    spans = filter_pipe(text)
    if not spans:
        return text, False

    redacted = text
    for s in sorted(spans, key=lambda x: x["start"], reverse=True):
        tag = f"[{s.get('entity_group', s.get('entity', 'PII')).upper()}]"
        redacted = redacted[: s["start"]] + tag + redacted[s["end"] :]

    return redacted, True

In [ ]:
_SENSITIVE_QUERIES = [[
    "face", "passport", "drivers license",
    "credit card", "medical prescription", "laptop screen",
]]


def detect_privacy_risks_from_image(
    image: Image.Image,
    use_ocr: bool = True,
) -> list[list[int]]:
    """Return deduplicated [xmin, ymin, xmax, ymax] sensitive boxes for one image.

    Detection steps:
    1. OwlViT zero-shot object detection for privacy-sensitive objects.
    2. (Optional) EasyOCR + text PII filter — bounding boxes only, strings
       deleted immediately after classification (ephemeral OCR principle).

    Boxes are expanded by 8 px and merged via IoU-based NMS before returning,
    reducing redundant inpainting regions.
    """
    image_w, image_h = image.size
    if image_w == 0 or image_h == 0:
        return []

    raw_boxes: list[list[int]] = []

    # ── 1. OwlViT visual object detection ─────────────────────────────────────
    inputs = owl_processor(
        text=_SENSITIVE_QUERIES,
        images=image,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = owl_model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = owl_processor.post_process_grounded_object_detection(
        outputs,
        threshold=0.1,
        target_sizes=target_sizes,
    )[0]

    for box in results["boxes"]:
        raw_boxes.append([int(v) for v in box.tolist()])

    # ── 2. Ephemeral OCR text detection ───────────────────────────────────────
    if use_ocr:
        ocr_results = reader.readtext(np.array(image))
        for bbox, ocr_text, _prob in ocr_results:
            if not ocr_text or not str(ocr_text).strip():
                continue
            _redacted, has_pii = redact_text(ocr_text, pii_filter)
            if has_pii:
                xs = [p[0] for p in bbox]
                ys = [p[1] for p in bbox]
                raw_boxes.append([int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))])
        del ocr_results  # never retained — ephemeral OCR strings leave no trace

    # ── 3. Expand + deduplicate ────────────────────────────────────────────────
    expanded = [expand_box(b, image_w, image_h) for b in raw_boxes]
    return nms_boxes(expanded)

In [ ]:
def detect_privacy_risks(
    user_text: str | None = None,
    image_path: str | None = None,
    use_ocr: bool = True,
) -> list[dict]:
    """Detect privacy risks across text and image/PDF inputs.

    Handles all five input combinations:
      - text + image  (with or without OCR)
      - text only
      - image only    (with or without OCR)
      - empty input

    Returns a list of per-page result dicts, each containing:
      {
        "image"          : PIL RGB image or None,
        "redacted_text"  : str  (PII replaced with [ENTITY_TYPE] tags),
        "sensitive_boxes": list of [xmin, ymin, xmax, ymax],
        "page_index"     : int or None,
        "source_path"    : str or None,
      }
    """
    results = []

    # ── Text PII detection (runs once, shared across all pages) ───────────────
    if user_text is not None and str(user_text).strip():
        redacted_user_text, _has_pii = redact_text(user_text, pii_filter)
    else:
        redacted_user_text = ""

    # ── Image / PDF loading ────────────────────────────────────────────────────
    images = load_images_from_path(image_path)

    # ── Text-only or empty input ───────────────────────────────────────────────
    if not images:
        results.append({
            "image": None,
            "redacted_text": redacted_user_text,
            "sensitive_boxes": [],
            "page_index": None,
            "source_path": image_path,
        })
        return results

    # ── Per-page image processing ──────────────────────────────────────────────
    for page_index, image in enumerate(images):
        sensitive_boxes = detect_privacy_risks_from_image(image, use_ocr=use_ocr)
        results.append({
            "image": image,
            "redacted_text": redacted_user_text,
            "sensitive_boxes": sensitive_boxes,
            "page_index": page_index,
            "source_path": image_path,
        })

    return results

## Phase 3 & 4: Policy Engine and Memory Write Gate

This function calculates the density of the privacy risks to determine the optimal stage-aware flow control policy:
* **ALLOW:** No risks detected. Store the raw input.
* **MASK (<30% Density):** Use local Stable Diffusion inpainting to erase the specific sensitive bounding boxes, preserving the rest of the image's utility for the CLIP embedding.
* **ABSTRACT (>30% Density):** Block the image entirely and replace it with a text-based semantic summary to prevent vector pollution.

In [ ]:
def embed_safe_memory(
    text: str | None = None,
    image: Image.Image | None = None,
) -> dict:
    """CLIP embedding for safe (post-gate) content.

    Handles all four valid combinations of text/image presence.
    Returns a dict:
      {
        "text_embeds" : Tensor | None,
        "image_embeds": Tensor | None,
      }

    Note: avoids get_text_features / get_image_features because in
    transformers 5.x those helpers return BaseModelOutputWithPooling
    rather than a plain tensor. We call the sub-models directly instead.
    """
    has_text = text is not None and str(text).strip()
    has_image = image is not None

    out: dict = {"text_embeds": None, "image_embeds": None}

    if not has_text and not has_image:
        return out

    if has_text and has_image:
        # Full joint forward pass — CLIPModel handles both modalities
        inputs = clip_processor(
            text=[str(text)], images=image, return_tensors="pt", padding=True,
        ).to(device)
        with torch.no_grad():
            result = clip_model(**inputs)
        out["text_embeds"] = result.text_embeds
        out["image_embeds"] = result.image_embeds

    elif has_text:
        # Text-only: text_model → pooler_output → text_projection
        inputs = clip_processor(
            text=[str(text)], return_tensors="pt", padding=True,
        ).to(device)
        with torch.no_grad():
            text_out = clip_model.text_model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
            )
            out["text_embeds"] = clip_model.text_projection(text_out.pooler_output)

    else:
        # Image-only: vision_model → pooler_output → visual_projection
        inputs = clip_processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            vision_out = clip_model.vision_model(pixel_values=inputs["pixel_values"])
            out["image_embeds"] = clip_model.visual_projection(vision_out.pooler_output)

    return out

In [ ]:
_INPAINT_SIZE = (512, 512)  # Stable Diffusion inpainting requires fixed dimensions


def privacy_gate_and_embed(
    image: Image.Image | None = None,
    redacted_text: str = "",
    sensitive_boxes: list | None = None,
) -> tuple:
    """Policy engine + memory write gate.

    Determines the privacy action for a single memory event and returns
    a safe embedding ready for vector DB insertion.

    Policy decisions:
      EMPTY         — no text or image provided
      TEXT_ONLY     — text present, no image
      ALLOW         — image has no sensitive regions
      MASK          — <30 % sensitive area: inpaint sensitive boxes
      ABSTRACT      — ≥30 % sensitive area: block image, store text summary
      INVALID_IMAGE — image has zero area

    Returns:
      (policy, final_image, abstract_summary, embeddings, redacted_text)
    """
    if sensitive_boxes is None:
        sensitive_boxes = []
    redacted_text = str(redacted_text) if redacted_text else ""

    has_text = bool(redacted_text.strip())
    has_image = image is not None

    # ── Case 0: nothing provided ───────────────────────────────────────────────
    if not has_text and not has_image:
        return (
            "EMPTY", None,
            "No valid text or image input was provided.",
            embed_safe_memory(), redacted_text,
        )

    # ── Case 1: text only ──────────────────────────────────────────────────────
    if has_text and not has_image:
        return (
            "TEXT_ONLY", None,
            "Text-only memory entry. No image was provided.",
            embed_safe_memory(text=redacted_text), redacted_text,
        )

    # ── Case 2: image present → apply image policy ─────────────────────────────
    image_w, image_h = image.size
    total_area = image_w * image_h

    if total_area == 0:
        memory_text = redacted_text if has_text else "Image has invalid size and was blocked."
        return (
            "INVALID_IMAGE", None,
            "Image has invalid size and was blocked.",
            embed_safe_memory(text=memory_text), redacted_text,
        )

    sensitive_area = sum(calculate_box_area(b) for b in sensitive_boxes)
    sensitive_ratio = sensitive_area / total_area

    # ── ALLOW ──────────────────────────────────────────────────────────────────
    if sensitive_ratio == 0:
        policy = "ALLOW"
        final_image = image
        summary = "Raw image allowed — no sensitive regions detected."

    # ── MASK (<30 %) ───────────────────────────────────────────────────────────
    elif sensitive_ratio < 0.30:
        policy = "MASK"
        print(f"  [MASK] Sensitive coverage {sensitive_ratio:.1%} — running local inpainting...")

        # Build binary mask (white = erase)
        mask = Image.new("RGB", image.size, (0, 0, 0))
        draw = ImageDraw.Draw(mask)
        for box in sensitive_boxes:
            draw.rectangle(box, fill=(255, 255, 255))

        # Stable Diffusion requires fixed input dimensions; restore after
        img_sd = image.resize(_INPAINT_SIZE, Image.LANCZOS)
        mask_sd = mask.resize(_INPAINT_SIZE, Image.NEAREST)

        inpainted = inpaint_pipe(
            prompt="neutral background, empty space, seamless texture",
            image=img_sd,
            mask_image=mask_sd,
            num_inference_steps=20,
        ).images[0]

        final_image = inpainted.resize(image.size, Image.LANCZOS)
        summary = f"Image masked via generative inpainting ({len(sensitive_boxes)} region(s))."

    # ── ABSTRACT (≥30 %) ───────────────────────────────────────────────────────
    else:
        policy = "ABSTRACT"
        print(f"  [ABSTRACT] Sensitive coverage {sensitive_ratio:.1%} — blocking image.")

        final_image = None
        summary = (
            "ABSTRACT SEMANTIC SUMMARY: Image contained a high density of "
            "sensitive personal features and was blocked from memory."
        )

    # ── Memory write: produce safe CLIP embedding ──────────────────────────────
    if final_image is not None:
        embeddings = embed_safe_memory(
            text=redacted_text if has_text else None,
            image=final_image,
        )
    else:
        memory_text = (redacted_text + " " + summary).strip() if has_text else summary
        embeddings = embed_safe_memory(text=memory_text)

    return policy, final_image, summary, embeddings, redacted_text

## Phase 5: Execution & Visualization

Run test payloads through the multimodal gate to verify text redaction,
image inpainting / abstraction, and safe CLIP embedding generation.

Three demo scenarios:
1. **Text + Image** — combined multimodal input
2. **Text only** — batch of PII-laden prompts, no image
3. **Image / PDF only** — visual-only inputs with and without OCR

In [ ]:
test_image_path  = "data/test_upload.jpg"
test_user_prompt = (
    "Save this to my memory: my new phone number is 555-0199 "
    "and here is a photo of my desk."
)

print("=" * 62)
print("  DEMO 1 — Text + Image")
print("=" * 62)

# ── Sensing Gate ──────────────────────────────────────────────────────────────
detection_results = detect_privacy_risks(test_user_prompt, test_image_path)
r        = detection_results[0]          # single image → one result
raw_image = r["image"]
safe_text = r["redacted_text"]
boxes     = r["sensitive_boxes"]

# ── Policy & Memory Gate ──────────────────────────────────────────────────────
policy, final_img, summary, embeddings, safe_text = privacy_gate_and_embed(
    image=raw_image,
    redacted_text=safe_text,
    sensitive_boxes=boxes,
)

print(f"\n  Policy enforced : {policy}")
print(f"  Original text   : {test_user_prompt}")
print(f"  Redacted text   : {safe_text}")
print(f"  System message  : {summary}")
print(f"  Sensitive boxes : {len(boxes)} region(s) detected")

# ── Visualization ─────────────────────────────────────────────────────────────
if raw_image is not None:
    n_panels = 3 if final_img is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

    axes[0].imshow(raw_image)
    axes[0].set_title("Original  (unsafe)", fontsize=12, color="#c0392b", fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(draw_boxes_on_image(raw_image, boxes))
    axes[1].set_title(f"Detected regions  ({len(boxes)})", fontsize=12)
    axes[1].axis("off")

    if final_img is not None:
        axes[2].imshow(final_img)
        axes[2].set_title(
            f"After gate  [{policy}]", fontsize=12, color="#27ae60", fontweight="bold",
        )
        axes[2].axis("off")

    plt.suptitle("Privacy Gate — Text + Image Demo", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("\n  [ ABSTRACT — image suppressed; not stored in memory ]")

# ── Embedding summary ─────────────────────────────────────────────────────────
print("\n  Embedding summary:")
if embeddings["image_embeds"] is not None:
    print(f"    Image : {tuple(embeddings['image_embeds'].shape)}")
if embeddings["text_embeds"] is not None:
    print(f"    Text  : {tuple(embeddings['text_embeds'].shape)}")

In [ ]:
test_prompts = [
    "Save this: I used my credit card to buy concert tickets. Card # 123-456-7890, CVV 215.",
    "My SSN is 102-93-8564.",
    "My ex-wife lives at 4000 Mayflower Hill Drive.",
]

print("=" * 62)
print("  DEMO 2 — Text Only (batch)")
print("=" * 62)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n  ── Test {i} {'─' * 48}")

    results = detect_privacy_risks(user_text=prompt, image_path=None)
    r = results[0]

    policy, _final_img, summary, embeddings, safe_text = privacy_gate_and_embed(
        image=r["image"],
        redacted_text=r["redacted_text"],
        sensitive_boxes=r["sensitive_boxes"],
    )

    print(f"  Policy   : {policy}")
    print(f"  Original : {prompt}")
    print(f"  Redacted : {safe_text}")
    if embeddings["text_embeds"] is not None:
        print(f"  Embedding: {tuple(embeddings['text_embeds'].shape)}")

In [ ]:
test_image_paths = [
    "data/test_upload.jpg",
    "data/test_codebase.png",
    "data/test_email.png",
    "data/test_prescription.pdf",
]

print("=" * 62)
print("  DEMO 3 — Image / PDF Only")
print("=" * 62)

for file_i, path in enumerate(test_image_paths, 1):
    fname = os.path.basename(path)
    print(f"\n\n  ── File {file_i}: {fname} {'─' * max(0, 44 - len(fname))}")

    detection_results = detect_privacy_risks(user_text=None, image_path=path)

    for r in detection_results:
        raw_image = r["image"]
        boxes     = r["sensitive_boxes"]
        page_idx  = r["page_index"]
        page_lbl  = f" (page {page_idx + 1})" if page_idx is not None else ""

        print(f"\n  Processing{page_lbl}…")

        policy, final_img, summary, embeddings, safe_text = privacy_gate_and_embed(
            image=raw_image,
            redacted_text=r["redacted_text"],
            sensitive_boxes=boxes,
        )

        print(f"  Policy          : {policy}")
        print(f"  Sensitive boxes : {len(boxes)}")
        print(f"  System message  : {summary}")

        if raw_image is None:
            print("  [ No valid image found ]")
            continue

        # ── Visualization ─────────────────────────────────────────────────────
        if final_img is not None:
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            axes[0].imshow(raw_image)
            axes[0].set_title("Original  (unsafe)", fontsize=12, color="#c0392b", fontweight="bold")
            axes[0].axis("off")

            axes[1].imshow(draw_boxes_on_image(raw_image, boxes))
            axes[1].set_title(f"Detected regions  ({len(boxes)})", fontsize=12)
            axes[1].axis("off")

            axes[2].imshow(final_img)
            axes[2].set_title(
                f"After gate  [{policy}]", fontsize=12, color="#27ae60", fontweight="bold",
            )
            axes[2].axis("off")
        else:
            # ABSTRACT — show original + detections only
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(raw_image)
            axes[0].set_title("Original  (unsafe)", fontsize=12, color="#c0392b", fontweight="bold")
            axes[0].axis("off")

            axes[1].imshow(draw_boxes_on_image(raw_image, boxes))
            axes[1].set_title(
                f"ABSTRACT — image blocked  ({len(boxes)} regions)",
                fontsize=12, color="#e67e22", fontweight="bold",
            )
            axes[1].axis("off")
            print("  [ ABSTRACT — image not stored in memory ]")

        plt.suptitle(f"{fname}{page_lbl}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

        # ── Embedding summary ─────────────────────────────────────────────────
        if embeddings["image_embeds"] is not None:
            print(f"  Image embedding : {tuple(embeddings['image_embeds'].shape)}")
        if embeddings["text_embeds"] is not None:
            print(f"  Text  embedding : {tuple(embeddings['text_embeds'].shape)}")